# Mendi Stream Cleaning

Extra Mendi-specific steam preprocessing steps involving removing ambience and detecting noise.

In [1]:
from scipy.stats import sem

COLS = ["ir_r", "red_r", "ir_l", "red_l", "ir_p", "red_p"]

def clean_stream(df):
    df_copy = df.copy()

    _remove_ambience(df_copy)
    _remove_movement(df_copy)

    # Interpolate any NaN values leftover from cleaning the data
    return df_copy.interpolate(method="linear").ffill().bfill()


def _remove_ambience(df):
    # Subtract ambience from the readings
    df['ir_l'] = df['ir_l'] - df['amb_l']
    df['red_l'] = df['red_l'] - df['amb_l']
    df['ir_r'] = df['ir_r'] - df['amb_r']
    df['red_r'] = df['red_r'] - df['amb_r']
    df['ir_p'] = df['ir_p'] - df['amb_p']
    df['red_p'] = df['red_p'] - df['amb_p']

    # Set negative values as invalid (NaN)
    df[COLS] = df[COLS].mask(df[COLS] < 0)


def _remove_movement(df, movement_threshold=0.1, g_scale=16384):
    # Convert raw units to G units
    acc_x = df['acc_x'] / g_scale
    acc_y = df['acc_y'] / g_scale
    acc_z = df['acc_z'] / g_scale

    # Calculate the scalar magnitude of the 3D accelerometer
    acc_magnitude = np.sqrt(acc_x ** 2 + acc_y ** 2 + acc_z ** 2)

    # Find samples with an acceleration greater than the threshold
    movement_mask = np.abs(acc_magnitude - 1.0) >= movement_threshold

    # Set movement rows as invalid (NaN)
    df.loc[movement_mask, COLS] = np.nan


# Optional Channel Plots

Helper methods to display plots for fNIRS channels other than what's being analysed for debugging purposes.

In [2]:
import plotly


def plot_accelerometer(df, g_scale=16384):
    time = np.arange(df.iloc[-1]["timestamp"])

    # Convert raw units to G units
    acc_x_g = df['acc_x'] / g_scale
    acc_y_g = df['acc_y'] / g_scale
    acc_z_g = df['acc_z'] / g_scale

    titles = (
        "Accelerometer X-axis",
        "Accelerometer Y-axis",
        "Accelerometer Z-axis"
    )

    fig = plotly.subplots.make_subplots(
        rows=3, cols=1,
        subplot_titles=titles,
        vertical_spacing=0.08,
        shared_xaxes=True
    )

    fig.add_trace(go.Scatter(
        x=time, y=acc_x_g, mode='lines', name='Acc X',
        line=dict(color='red'), showlegend=True
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=time, y=acc_y_g, mode='lines', name='Acc Y',
        line=dict(color='green'), showlegend=True
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=time, y=acc_z_g, mode='lines', name='Acc Z',
        line=dict(color='blue'), showlegend=True
    ), row=3, col=1)

    fig.show()


# Data Preprocessing

Converts the raw Mendi signals into clean HbO/Hb concentration signals and writes them to a SNIRF file. The following pipeline is applied:

1. Data cleaning - the events dataframe is organised to have a standard ordering. Ambience and motion detected in the stream is corrected using interpolation.
2. Optical Density Conversion - the raw light intensity readings are converted into optical density (OD) readings using an initial rest baseline from the events dataframe.
3. HbO Conversion - Each specified pair of optical density channels are converted into HbO/Hb readings by applying the modified beer-lambert law (MBLL).
4. IIR Filtering - Each new HbO/Hb channel is immediately passed through a 6th order Butterworth IIR filter with a cutoff frequency of 0.02Hz to match the paper.
5. Short Channel Regression - The Mendi pulse short channel is used as a short channel regressor in order to compute the residual in the remaining long channels.
6. Median Normalisation - The HbO/Hb dataset is median normalised to reduce the impact of spikes that skew the mean when analysing the haemodynamic response.
7. SNIRF Writing - The participants fully preprocessed HbO/Hb dataset is then written to its own SNIRF file located under the data/preprocessed directory.

In [3]:
import numpy as np
import pandas as pd

import mw_analysis.utils as utils
import mw_analysis.snirf as snirf

CHANNELS = [
    "acc_x", "acc_y", "acc_z",
    "ang_x", "ang_y", "ang_z",
    "temp",
    "ir_l", "red_l", "amb_l",
    "ir_r", "red_r", "amb_r",
    "ir_p", "red_p", "amb_p",
    "battery_voltage", "timestamp"
]

MAPPINGS = {
    "s1_d1": ("red_l", "ir_l"),
    "s1_d2": ("red_r", "ir_r"),
    "s1_d3": ("red_p", "ir_p"),
}

# Processed datasets
datasets = []

for subject_id in range(1, 9):
    stream_df, events_df = utils.nirscord_h5_to_df(f"../data/participant{subject_id}-mendi.h5", CHANNELS)

    # Clean the dataframes
    stream_df = clean_stream(stream_df)
    events_df = utils.clean_events(events_df)

    # Find the time when the experiment starts
    timestamp = utils.find_experiment_start(events_df)
    stream_df = stream_df.loc[stream_df["timestamp"] >= timestamp]
    events_df = events_df.loc[events_df["timestamp"] >= timestamp]

    # Zero-base the stream and event timestamps
    stream_df["timestamp"] -= timestamp
    events_df["timestamp"] -= timestamp

    # Convert to optical density
    utils.od(stream_df, events_df, COLS)

    # Convert to HbO/Hb
    hbo_df = stream_df[["timestamp"]]

    for name, (ch1, ch2) in MAPPINGS.items():
        HbO, Hb = utils.mbll(stream_df[ch1].values, stream_df[ch2].values)
        hbo_df[f"{name} hbo"] = utils.iir_filter(HbO)
        hbo_df[f"{name} hb"] = utils.iir_filter(Hb)

    # Extract short channel regressor columns
    hbo_cols = hbo_df.filter(regex="hbo$").columns.tolist()
    hb_cols = hbo_df.filter(regex="hb$").columns.tolist()

    # Apply short channel regressor to HbO
    hbo_df[hbo_cols[:-1]] = utils.short_channel_regressor(
        hbo_df[hbo_cols[:-1]],
        hbo_df[hbo_cols[-1]]
    )

    # Apply short channel regressor to Hb
    hbo_df[hb_cols[:-1]] = utils.short_channel_regressor(
        hbo_df[hb_cols[:-1]],
        hbo_df[hb_cols[-1]]
    )

    # Median normalise the columns
    hbo_df[hbo_cols + hb_cols] -= hbo_df[hbo_cols + hb_cols].median()

    # Combine the datasets
    dataset = snirf.NirscordDataset(subject_id, hbo_df, events_df)

    # Write to SNIRF file
    snirf.to_snirf(dataset, f"../data/processed/participant{subject_id}-mendi.snirf")

    # Cache the dataset for analysis
    datasets.append(dataset)


# Participant HbO Concentrations

Displays the HbO concentrations from every channel in different colours for each participant.

In [4]:
import plotly.graph_objects as go

colours = ["red", "green", "blue"]

for dataset in datasets:
    fig = go.Figure()

    # Generate a set of time values across the whole experiment
    time = np.arange(dataset.stream_df.iloc[-1]["timestamp"])

    # Plot every channel as its own individually coloured line
    for i, name in enumerate(MAPPINGS.keys()):
        fig.add_trace(go.Scatter(
            x=time, y=dataset.stream_df[f"{name} hbo"],
            mode='lines', name=f'{name} Δ[HbO]',
            line=dict(color=colours[i]),
            hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
        ))
    fig.update_layout(
        title=f'Participant {dataset.subject_id} HbO concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )
    fig.show()

# Average SART (No) Error Epoch

Displays the average SART (No) Error epoch across the entire dataset ranging from -30s to 10s for each channel.

In [5]:
# Generate a set of time values in the epoch range to display
time = np.arange(-30, 10)

# Collect all SART (No) error epochs into two separate lists
no_error_epochs, error_epochs = utils.collect_epochs(datasets, range(-30, 10))

# Compute the average and sem of each SART (No) Error cell
error_group = pd.concat(error_epochs).groupby(level=0)
avg_sart_errors = error_group.mean().dropna()
sem_sart_errors = error_group.sem().dropna()

no_error_group = pd.concat(no_error_epochs).groupby(level=0)
avg_sart_no_errors = no_error_group.mean().dropna()
sem_sart_no_errors = no_error_group.sem().dropna()

for i, name in enumerate(MAPPINGS.keys()):
    fig = go.Figure()

    # Calculate the error mean and uncertainty
    error_mean = avg_sart_errors[f"{name} hbo"]
    error_sem = sem_sart_errors[f"{name} hbo"]

    upper_error = error_mean + error_sem
    lower_error = error_mean - error_sem

    # Display the error mean line
    fig.add_trace(go.Scatter(
        x=time,
        y=error_mean,
        mode='lines',
        name='SART Error',
        line=dict(color='rgb(0, 0, 255)'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Display the error sem area
    fig.add_trace(go.Scatter(
        x=time,
        y=lower_error,
        mode='lines',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig.add_trace(go.Scatter(
        x=time,
        y=upper_error,
        mode='lines',
        fill='tonexty',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        fillcolor='rgba(0, 0, 255, 0.2)',
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Calculate the no error mean and uncertainty
    no_error_mean = avg_sart_no_errors[f"{name} hbo"]
    no_error_sem = sem_sart_no_errors[f"{name} hbo"]

    upper_no_error = no_error_mean + no_error_sem
    lower_no_error = no_error_mean - no_error_sem

    # Display the no error mean line
    fig.add_trace(go.Scatter(
        x=time,
        y=no_error_mean,
        mode='lines',
        name='SART No Error',
        line=dict(color='rgb(0, 255, 0)'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Display the no error sem area
    fig.add_trace(go.Scatter(
        x=time,
        y=lower_no_error,
        mode='lines',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig.add_trace(go.Scatter(
        x=time,
        y=upper_no_error,
        mode='lines',
        fill='tonexty',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        fillcolor='rgba(0, 255, 0, 0.2)',
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Add title and metadata to plot
    fig.update_layout(
        title=f'{name.upper()} Average SART (No) Error HbO Concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )

    fig.show()


# Participant LDA Classifiers

Performs 10-fold cross validation on an LDA classifier trained from each participants dataset.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Stat tracking
accuracies = []
std_errors = []

print("=" * 40)
print("10-fold cross validation")
print("=" * 40)

for dataset in datasets:
    no_error_epochs, error_epochs = utils.extract_epochs(
        dataset.stream_df, dataset.events_df, range(-15, -5)
    )

    # Calculate the averages of both no error and error epochs
    no_error_epoch_means = pd.DataFrame(
        [epoch.mean() for epoch in no_error_epochs]
    ).dropna(how="all") # Empty rows occur when recording stopped

    error_epoch_means = pd.DataFrame(
        [epoch.mean() for epoch in error_epochs]
    ).dropna(how="all") # Empty rows occur when recording stopped

    # Extract the healthy HbO columns
    X = pd.concat([no_error_epoch_means, error_epoch_means])
    X = X.filter(like="hbo").dropna(axis=1, how="all")

    # Drop the unwanted short channels
    X = X.drop(columns=["s1_d3 hbo"])

    # Check the input data is able to be 10-fold cross validated
    if len(no_error_epoch_means) < 10:
        print(
            f"Subject {dataset.subject_id} | "
            f"Class no error only has {len(no_error_epoch_means)} samples"
        )
        continue

    elif len(error_epoch_means) < 10:
        print(
            f"Subject {dataset.subject_id} | "
            f"Class error only has {len(error_epoch_means)} samples"
        )
        continue

    # Extract the input class labels
    Y = np.concatenate([
        np.zeros(len(no_error_epoch_means), dtype=int),
        np.ones(len(error_epoch_means), dtype=int)
    ])

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, # The average epoch HbO concentrations
        Y, # Corresponding SART (No) Error labels
        test_size=0.2,
        stratify=Y,
        random_state=42
    )

    # Define the LDA processing and training pipeline
    lda = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lda", LinearDiscriminantAnalysis())
    ])

    # Define the k-fold cross validation configuration
    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Perform k-fold cross validation on the pipeline
    cv_scores = cross_val_score(
        lda,
        X_train,
        Y_train,
        cv=cv,
        scoring="accuracy"
    )

    # Compute the average accuracy and standard error
    mean_acc = cv_scores.mean()
    std_err = sem(cv_scores)

    print(
        f"Subject {dataset.subject_id} |",
        f"CV Accuracy: {mean_acc:.1%} ± {std_err:.1%} SE"
    )

    accuracies.append(mean_acc)
    std_errors.append(std_err)

# Compute the average accuracy across all participants
mean_accuracy = np.mean(accuracies)
mean_std_error = np.mean(std_errors)

print("=" * 40)
print(f"Mean Accuracy: {mean_accuracy:.1%}")
print(f"Mean Standard Error: {mean_std_error:.1%}")
print("=" * 40)


10-fold cross validation
Subject 1 | CV Accuracy: 72.5% ± 2.5% SE
Subject 2 | Class error only has 9 samples
Subject 3 | CV Accuracy: 57.5% ± 5.3% SE
Subject 4 | CV Accuracy: 59.2% ± 6.9% SE
Subject 5 | CV Accuracy: 45.0% ± 4.8% SE
Subject 6 | CV Accuracy: 50.0% ± 7.5% SE
Subject 7 | CV Accuracy: 50.0% ± 7.5% SE
Subject 8 | CV Accuracy: 54.2% ± 4.2% SE
Mean Accuracy: 55.5%
Mean Standard Error: 5.5%


# Collected LDA Classifier

Trains and tests an LDA classifier created from collecting all participant datasets together.

In [7]:
import plotly.express as px

# Collect all SART (No) error epochs into two separate lists
no_error_epochs, error_epochs = utils.collect_epochs(datasets, range(-15, -5))

# Calculate the averages of both no error and error epochs
no_error_epoch_means = pd.DataFrame(
    [epoch.mean() for epoch in no_error_epochs]
).dropna(how="all") # Empty rows occur when recording stopped

error_epoch_means = pd.DataFrame(
    [epoch.mean() for epoch in error_epochs]
).dropna(how="all") # Empty rows occur when recording stopped

# Extract the healthy HbO columns
X = pd.concat([no_error_epoch_means, error_epoch_means])
X = X.filter(like="hbo").dropna(axis=1, how="all")

# Drop the unwanted short channels
X = X.drop(columns=["s1_d3 hbo"])

# Extract the input class labels
Y = np.concatenate([
    np.zeros(len(no_error_epoch_means), dtype=int),
    np.ones(len(error_epoch_means), dtype=int)
])

X_train, X_test, Y_train, Y_test = train_test_split(
    X, # The average epoch HbO concentrations
    Y, # Corresponding SART (No) Error labels
    test_size=0.2,
    stratify=Y,
    random_state=42
)

# Define the LDA processing and training pipeline
lda = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis())
])

# Train the LDA and predict using the test data
lda.fit(X_train, Y_train)
Y_pred = lda.predict(X_test)

# Plot the confusion matrix from the predictions
fig = px.imshow(
    confusion_matrix(Y_test, Y_pred),
    text_auto=True,
    x=["No Error", "Error"],
    y=["No Error", "Error"],
    color_continuous_scale="Blues",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},

)
fig.update_layout(
    title="LDA Confusion Matrix",
    width=600,
    height=500
)
fig.show()
